# Does the roughness gap drive L(b)?

$L(b) = EER(GS_b) - EER(GO_b)$ assumes the joins of $GS_b$ and $GO_b$ are equally rough, so their artefacts cancel. `join_discontinuity` (see `temporal-probes.ipynb`) shows GS's joins are somewhat rougher. This notebook checks whether that matters for the EER:

1. Per utterance, the mean distance between the frames at the joins of exactly the windows that were embedded (`src/evaluation/temporal_probes.py`'s `utterance_join_roughness`, same seed as the embeddings), for $GO_b$ and $GS_b$.
2. Per trial, the **roughness gap**: the mean over its two utterances of $GS_b$ minus $GO_b$ roughness.
3. Trials binned into quantiles of the gap, and $L(b)$ per bin with a paired trial-bootstrap interval (`src/experiment.py`'s `sst_by_roughness_gap`).

**Reading it:** if $L(b)$ stays flat across the bins (slope ≈ 0, intercept ≈ overall $L(b)$), the rougher GS joins aren't what drives $L(b)$. If it rises with the gap, and the intercept at gap 0 is much smaller than the overall $L(b)$, part of $L(b)$ is artefact. If (nearly) every trial has a positive gap, the value at gap 0 is an extrapolation of the fit, not an observed bin.

**Caveat:** the gap isn't assigned at random. Utterances with rough joins also differ in other ways (more spectral change, noisier recordings), which may change $L(b)$ on their own. A flat curve is the informative outcome; a slope is consistent with an artefact, but doesn't prove one.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import ExperimentConfig
from src.data.voxceleb import VoxCelebCorpus
from src.evaluation.temporal_probes import utterance_join_roughness
from src.evaluation.verification import eer_from_scores, trial_list_scores
from src.experiment import _lazy_trial_utterances, sst_by_roughness_gap
from src.models.registry import build_model
from src.training.trainer import extract_embeddings_by_strategy

if "SLURM_JOB_ID" in os.environ:  # data staged by slurm/stage-voxceleb.submit
    stage_dir = f"/scratch/{os.environ['SLURM_JOB_ID']}"
    os.environ["VOXCELEB_ROOT"] = f"{stage_dir}/voxceleb1"
    os.environ["VOXCELEB2_ROOT"] = f"{stage_dir}/voxceleb2"

In [2]:
CONFIG = "configs/pretrained-resnet34-voxceleb-2s.yaml"
block_sizes = [8, 16, 32, 64]
run_idx = 0          # seed of the test draws, as run_idx of the final evaluation
num_bins = 5         # quantile bins of the roughness gap
num_resamples = 200  # trial-bootstrap resamples per bin
device = "cuda" if torch.cuda.is_available() else "cpu"

config = ExperimentConfig.from_yaml(ROOT / CONFIG)
transformation = config.transformation
segment_length = config.data.segment_length(transformation)
gap_ratio = config.evaluation.sst_gap_ratio
model = build_model(config).to(device).eval()
corpus = VoxCelebCorpus(config)
trial_pairs = corpus.dev_trial_pairs  # VoxCeleb1-O: 37,611 trials over 4,708 utterances
utterances = _lazy_trial_utterances(corpus, trial_pairs, transformation)
strategies = [f"{prefix}{b}" for b in block_sizes for prefix in ("GO", "GS")]
print(f"{len(utterances)} utterances, {len(trial_pairs)} trials, segment {segment_length} frames, gap ratio {gap_ratio}")

/home/beso/Development/sage-diarization/Speaker-Recognition-Supra-Segmental-Temporal-Features/.venv/lib64/python3.14/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


==> VoxCeleb1: 153516 utterances (cached index /home/beso/.cache/datasets/voxceleb1/.sst-wav-index.tsv)
4708 utterances, 37611 trials, segment 196 frames, gap ratio 1.0


In [3]:
embeddings = extract_embeddings_by_strategy(
    model, utterances, segment_length, {s: np.random.default_rng(run_idx) for s in strategies},
    device=device, num_workers=config.training.num_workers, log_every=1000, gap_ratio=gap_ratio,
)
roughness, roughness_ids = utterance_join_roughness(utterances, segment_length, strategies, seed=run_idx, gap_ratio=gap_ratio)
assert roughness_ids == list(embeddings[strategies[0]][1])  # one roughness per embedding row

row = {utterance_id: i for i, utterance_id in enumerate(roughness_ids)}
idx1 = np.array([row[id1] for _, id1, _ in trial_pairs])
idx2 = np.array([row[id2] for _, _, id2 in trial_pairs])

/home/beso/Development/sage-diarization/Speaker-Recognition-Supra-Segmental-Temporal-Features/.venv/lib64/python3.14/site-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/beso/Development/sage-diarization/Speaker-Recognition-Supra-Segmental-Temporal-Features/.venv/lib64/python3.14/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader

KeyboardInterrupt: 

In [ ]:
strata, overall = {}, {}
for b in block_sizes:
    labels, go_scores = trial_list_scores(*embeddings[f"GO{b}"], trial_pairs)
    _, gs_scores = trial_list_scores(*embeddings[f"GS{b}"], trial_pairs)
    utterance_gap = roughness[f"GS{b}"] - roughness[f"GO{b}"]
    trial_gap = (utterance_gap[idx1] + utterance_gap[idx2]) / 2
    scored = np.isfinite(go_scores) & np.isfinite(gs_scores) & np.isfinite(trial_gap)
    overall[b] = eer_from_scores(labels[scored], gs_scores[scored]) - eer_from_scores(labels[scored], go_scores[scored])
    strata[b] = sst_by_roughness_gap(
        labels, go_scores, gs_scores, trial_gap, num_bins, num_resamples, np.random.default_rng(run_idx)
    )
    slope, intercept = np.polyfit(strata[b]["gap"], strata[b]["L"], 1)
    strata[b]["fit"] = (slope, intercept)
    print(f"b={b:>2}: {scored.sum()} trials, {np.mean(trial_gap[scored] <= 0):.1%} with gap <= 0; "
          f"L(b) overall {overall[b] * 100:.2f} %-points, at gap 0 (fit) {intercept * 100:.2f}")
    for gap, cost, low, high, n in zip(*(strata[b][k] for k in ("gap", "L", "ci_low", "ci_high", "trials"))):
        print(f"      gap {gap:7.3f}: L {cost * 100:6.2f} [{low * 100:6.2f}, {high * 100:6.2f}]  ({n} trials)")

In [ ]:
block_ms = {b: (b - 1) * transformation.frame_step_s * 1000 + transformation.frame_length_s * 1000 for b in block_sizes}
fig, axes = plt.subplots(1, len(block_sizes), figsize=(4 * len(block_sizes), 3.5), sharey=True, squeeze=False)
for ax, b in zip(axes[0], block_sizes):
    s = strata[b]
    ax.errorbar(s["gap"], s["L"] * 100, yerr=[(s["L"] - s["ci_low"]) * 100, (s["ci_high"] - s["L"]) * 100],
                fmt="o", capsize=3, label="L(b) per bin, 95% CI")
    slope, intercept = s["fit"]
    xs = np.linspace(min(0, s["gap"].min()), s["gap"].max(), 50)
    ax.plot(xs, (intercept + slope * xs) * 100, "--", label="linear fit")
    ax.axhline(overall[b] * 100, color="gray", linestyle=":", label="L(b), all trials")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set(title=f"b = {b} ({block_ms[b]:.0f} ms)", xlabel="roughness gap GS - GO")
    ax.grid(alpha=0.3, linestyle="--")
axes[0][0].set_ylabel("L(b) = EER(GS) - EER(GO) [%-points]")
axes[0][-1].legend(fontsize="small")
fig.suptitle(f"L(b) by roughness gap, {config.model.type}")
fig.tight_layout()
plt.show()